# Emotions dataset -> binary LTS pipeline (GitHub + Colab)

This notebook follows the same LTS run structure used in the 20 Newsgroups workflow:
1) prepare a binary task,
2) run `main_cluster` style active learning,
3) evaluate the saved checkpoint.

Prepared CSVs store **raw** `dair-ai/emotion` ids (`0` sadness … `5` surprise). Training maps your `-positive_label` / `TARGET_LABEL` to binary gold (`1` = target class).

Binary task after mapping:
- `1` = target emotion (`love` by default)
- `0` = all other emotions


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import sys

REPO_URL = "https://github.com/ShenyaoZhang/DS-GA-3001-Data-Engineering-Project.git"
BRANCH = "main"  # Change if your binary workflow lives on another branch

DRIVE_ROOT = "/content/drive/MyDrive"
REPO_ROOT = os.path.join(DRIVE_ROOT, "DS-GA-3001-Data-Engineering-Project")
EXPERIMENT_ROOT = os.path.join(REPO_ROOT, "emotions_rec")

SRC_DIR = os.path.join(EXPERIMENT_ROOT, "src")
PROMPTS_DIR = os.path.join(EXPERIMENT_ROOT, "prompts")
DATA_PROCESSED_DIR = os.path.join(EXPERIMENT_ROOT, "data", "processed")

if not os.path.exists(REPO_ROOT):
    !git clone -b {BRANCH} {REPO_URL} "{REPO_ROOT}"
else:
    %cd "{REPO_ROOT}"
    !git fetch origin
    !git checkout {BRANCH}
    !git pull

%cd "{EXPERIMENT_ROOT}"
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("Experiment root:", EXPERIMENT_ROOT)


In [ ]:
from getpass import getpass
import os

token = getpass("Enter your HF token: ")
os.environ["HF_TOKEN"] = token
os.environ["HUGGINGFACE_HUB_TOKEN"] = token
print("HF token set.")


In [ ]:
!pip -q install -r "{os.path.join(REPO_ROOT, 'requirements.txt')}"
!pip -q install datasets


In [ ]:
TARGET_LABEL = "love"

%cd "{EXPERIMENT_ROOT}"
!python -u scripts/prepare_emotions_binary.py --label {TARGET_LABEL}

TRAIN_FILE = f"data/processed/emotions_{TARGET_LABEL}_smoke_train"
VAL_FILE = f"data/processed/emotions_{TARGET_LABEL}_smoke_validation.csv"
TEST_FILE = f"data/processed/emotions_{TARGET_LABEL}_test.csv"

print("Train:", TRAIN_FILE)
print("Val:  ", VAL_FILE)
print("Test: ", TEST_FILE)


In [ ]:
import os

import pandas as pd

from labeling import EMOTION_MAP

train_df = pd.read_csv(TRAIN_FILE + ".csv")
val_df = pd.read_csv(VAL_FILE)
test_df = pd.read_csv(TEST_FILE)

target_id = EMOTION_MAP[TARGET_LABEL]


def binary_counts(series):
    return series.astype(int).apply(lambda x: 1 if x == target_id else 0).value_counts().to_dict()


print("Raw emotion id counts (0–5) — train:", train_df["label"].value_counts().sort_index().to_dict())
print("Binary (1=target) — train:", binary_counts(train_df["label"]))
print("Raw emotion id counts — val:", val_df["label"].value_counts().sort_index().to_dict())
print("Binary (1=target) — val:", binary_counts(val_df["label"]))
print("Raw emotion id counts — test:", test_df["label"].value_counts().sort_index().to_dict())
print("Binary (1=target) — test:", binary_counts(test_df["label"]))

FULL_TRAIN_CSV = os.path.join(DATA_PROCESSED_DIR, f"emotions_{TARGET_LABEL}_train.csv")
FEW_SHOT_PATH = os.path.join(PROMPTS_DIR, f"few_shot_examples_emotion_{TARGET_LABEL}.json")
!python -u scripts/build_few_shot_emotions.py --train_csv "{FULL_TRAIN_CSV}" --out "{FEW_SHOT_PATH}" --n_per_class 2
print("Few-shot path:", FEW_SHOT_PATH)


In [ ]:
EXPECTED = [
    "preprocessing.py",
    "LDA.py",
    "labeling.py",
    "random_sampling.py",
    "thompson_sampling.py",
    "fine_tune.py",
    "main_cluster_emotion_binary.py",
    "eval_emotion_binary.py",
]

missing = []
for name in EXPECTED:
    path = os.path.join(SRC_DIR, name)
    ok = os.path.exists(path)
    print(f"{name}: {'FOUND' if ok else 'MISSING'}")
    if not ok:
        missing.append(name)

script_ok = os.path.exists(os.path.join(EXPERIMENT_ROOT, "scripts", "prepare_emotions_binary.py"))
print("scripts/prepare_emotions_binary.py:", "FOUND" if script_ok else "MISSING")
if not script_ok:
    missing.append("scripts/prepare_emotions_binary.py")

build_ok = os.path.exists(os.path.join(EXPERIMENT_ROOT, "scripts", "build_few_shot_emotions.py"))
print("scripts/build_few_shot_emotions.py:", "FOUND" if build_ok else "MISSING")
if not build_ok:
    missing.append("scripts/build_few_shot_emotions.py")

few_shot_repo = os.path.join(PROMPTS_DIR, f"few_shot_examples_emotion_{TARGET_LABEL}.json")
print(f"{few_shot_repo}:", "FOUND" if os.path.exists(few_shot_repo) else "MISSING")
if not os.path.exists(few_shot_repo):
    missing.append(few_shot_repo)

if missing:
    raise FileNotFoundError(f"Missing: {missing}")


In [ ]:
import subprocess

targets = [os.path.join(SRC_DIR, f) for f in EXPECTED]
res = subprocess.run(["python", "-m", "py_compile", *targets], capture_output=True, text=True, cwd=EXPERIMENT_ROOT)
if res.returncode == 0:
    print("All files compiled successfully.")
else:
    print(res.stderr)


In [ ]:
%cd "{EXPERIMENT_ROOT}"

# main_cluster-style command with a short budget (~30 min target)
!python -u src/main_cluster_emotion_binary.py \
  -sample_size 200 \
  -filename "{TRAIN_FILE}" \
  -val_path "{VAL_FILE}" \
  -balance False \
  -sampling thompson \
  -filter_label True \
  -model_finetune bert-base-uncased \
  -labeling qwen \
  -model text \
  -baseline 0.10 \
  -metric f1_pos \
  -cluster_size 8 \
  -positive_label "{TARGET_LABEL}" \
  -few_shot_path "{FEW_SHOT_PATH}" \
  -hf_model_id "Qwen/Qwen2.5-3B-Instruct" \
  -max_iterations 3 \
  -num_train_epochs 2 \
  -max_length 128 \
  -batch_size 16 \
  -confidence_threshold 0.40


## Evaluate the trained binary model

After training, pick your best saved checkpoint from `models/`.

This prints:
- positive-class precision/recall/F1
- macro-F1
- accuracy
- mean confidence

In [ ]:
%cd "{EXPERIMENT_ROOT}"
!ls -la models

In [ ]:
%cd "{EXPERIMENT_ROOT}"
!python -u src/eval_emotion_binary.py \
  -test_path "{TEST_FILE}" \
  -model_path "models/binary_love_fine_tunned_0_bandit_0" \
  -target_emotion "{TARGET_LABEL}" \
  -base_model "bert-base-uncased" \
  -max_length 128